# Melanoma Detection — Transfer Learning with MobileNetV2

This notebook implements the pipeline:

1. Load and prepare data (resize, normalize, split, augment)
2. Load MobileNetV2 base (pretrained on ImageNet, frozen)
3. Add a classification head (pooling, dropout, dense output)
4. Train the model (compile, fit, monitor validation loss)
5. Evaluate the model (confusion matrix, ROC curve, AUC)

**Assumption about your data:** your Google Drive folder `AIMI Dataset Images` is expected to contain
one subfolder per class, e.g.:

```
AIMI Dataset Images/
    benign/
        img001.jpg
        ...
    malignant/
        img002.jpg
        ...
```

This is what `tf.keras.utils.image_dataset_from_directory` needs to infer labels automatically. If your
folder is flat (all images in one folder with labels in a CSV instead), let me know and the data-loading
cell below needs to be swapped for a CSV-driven loader — everything after it stays the same.


## 0. Mount Google Drive & imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, classification_report

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 1. Load and prepare data
Resize, normalize, split, augment

- **Resize**: all images are resized to `IMG_SIZE` on load.
- **Normalize**: pixel scaling is done later, inside the model, via `mobilenet_v2.preprocess_input`
  (this keeps the exact preprocessing MobileNetV2 was trained with, and means raw 0-255 images can be
  saved/inspected before normalization).
- **Split**: 70% train / 15% validation / 15% test, done by first splitting off 30% as a held-out set,
  then splitting that in half into validation and test.
- **Augment**: random flip/rotation/zoom/contrast, applied only to the training set.


In [ ]:
DATA_DIR = '/content/drive/MyDrive/AIMI Dataset Images'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

# 70% train
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.3,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
)

# 30% held out -> split into validation + test below
holdout_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.3,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
)

class_names = train_ds.class_names
print("Classes:", class_names)

holdout_batches = holdout_ds.cardinality().numpy()
val_ds = holdout_ds.take(holdout_batches // 2)
test_ds = holdout_ds.skip(holdout_batches // 2)

print(f"Train batches: {train_ds.cardinality().numpy()}, "
      f"Val batches: {val_ds.cardinality().numpy()}, "
      f"Test batches: {test_ds.cardinality().numpy()}")


In [ ]:
# Preview a few images
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i][0])])
        plt.axis("off")


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

AUTOTUNE = tf.data.AUTOTUNE

# Augmentation is applied only to the training set. Validation/test stay untouched
# so evaluation reflects real, unaugmented images.
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE,
).prefetch(AUTOTUNE)

val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


## 2. Load MobileNetV2 base
Pretrained on ImageNet, frozen


In [ ]:
IMG_SHAPE = IMG_SIZE + (3,)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # freeze the pretrained base

base_model.summary()


## 3. Add classification head
Pooling, dropout, dense output


In [ ]:
inputs = tf.keras.Input(shape=IMG_SHAPE)

# MobileNetV2's own preprocessing (scales pixels to [-1, 1])
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)  # binary: benign vs malignant

model = tf.keras.Model(inputs, outputs)
model.summary()


## 4. Train the model
Compile, fit, monitor validation loss


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/melanoma_mobilenetv2_best.keras',
        monitor='val_loss', save_best_only=True,
    ),
]

EPOCHS = 20

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(loss))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Loss')

plt.show()


## 5. Evaluate the model
Confusion matrix, ROC curve, AUC

Evaluation runs on `test_ds` — the held-out split the model never saw during training or
validation-based early stopping.


In [ ]:
test_loss, test_acc, test_auc = model.evaluate(test_ds)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f} | Test AUC: {test_auc:.4f}")


In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0).flatten()
y_pred_prob = model.predict(test_ds).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()


## 6. Try it: minimal image-upload front end (Gradio)

A small web UI, embedded right in this notebook, for testing one image at a time: upload a lesion
image and get back a predicted class with a confidence score.

**Disclaimer:** this is a demo interface for a student/research project, not a medical device. It must
not be used to make real diagnostic decisions, and `share=True` below creates a temporary *public* link
— don't upload real patient data to it.


In [ ]:
!pip install -q gradio


In [ ]:
# If you're running this section in a fresh runtime (no training done this session),
# load the model you saved during training instead of retraining:
# model = tf.keras.models.load_model('/content/drive/MyDrive/melanoma_mobilenetv2_best.keras')


In [ ]:
import gradio as gr
from PIL import Image


def predict_image(img):
    if img is None:
        return None

    image = Image.fromarray(img).convert('RGB').resize(IMG_SIZE)
    arr = np.expand_dims(np.array(image, dtype=np.float32), axis=0)  # (1, H, W, 3), pixels 0-255

    prob_class1 = float(model.predict(arr, verbose=0)[0][0])  # model already includes its own preprocessing
    return {
        class_names[0]: 1 - prob_class1,
        class_names[1]: prob_class1,
    }


demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type='numpy', label='Upload a skin lesion image'),
    outputs=gr.Label(num_top_classes=2, label='Prediction'),
    title='Melanoma Detection (Demo)',
    description=(
        'Upload a skin lesion image to get a classification and confidence score. '
        'Educational/demo use only — not a medical diagnostic tool.'
    ),
)

demo.launch(share=True, debug=False)


## Optional: Fine-tuning (beyond the flowchart)

Not part of the 5-step flowchart, but a common next step once the frozen-base model plateaus: unfreeze the
top layers of MobileNetV2 and continue training with a low learning rate for a few more epochs. Only run
this after step 4/5 above, and re-run evaluation afterward if you use it.


In [ ]:
# base_model.trainable = True
#
# fine_tune_at = 100  # keep the first 100 layers frozen, unfreeze the rest
# for layer in base_model.layers[:fine_tune_at]:
#     layer.trainable = False
#
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
#     loss='binary_crossentropy',
#     metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
# )
#
# history_fine = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=10,
#     callbacks=callbacks,
# )
